# Calenda — Qwen3-0.6B LoRA 학습 (Colab · 단일 GPU)

단일 베이스 **Qwen/Qwen3-0.6B** · **completion-only 손실 ON** (프롬프트 마스킹, 응답 토큰만 학습).

**GPU 우선순위:** H100 / A100 / L4 권장. **T4는 bf16 미지원(fp16 강제 → 품질 손실)이라 비권장.**

데이터·코드는 repo에 포함 (별도 업로드 불필요). 위→아래 순서 실행.

## 0. 환경 초기화 (가장 먼저 실행)

In [ ]:
import os

# HF_TOKEN: Colab Secrets에 등록돼 있으면 읽고, 없으면 빈값(공개 모델이므로 학습/merge는 무관)
try:
    from google.colab import userdata
    _hf_token = userdata.get('HF_TOKEN') or ''
except Exception:
    _hf_token = ''

os.environ['HF_TOKEN'] = _hf_token
os.environ['HUGGINGFACE_HUB_TOKEN'] = _hf_token
os.environ['HF_HUB_DISABLE_IMPLICIT_TOKEN'] = '1'   # 자동 Secrets 조회 팝업 억제
os.environ['WANDB_DISABLED'] = 'true'

if _hf_token:
    print(f'HF_TOKEN 로드됨 ({len(_hf_token)}자) — 어댑터 HF 업로드 가능')
else:
    print('HF_TOKEN 없음 — 학습/평가는 정상, HF 업로드 셀은 건너뜀')

## 1. GPU 확인

In [ ]:
!nvidia-smi

## 2. repo clone + 버전 배너

In [ ]:
import os, yaml, json, subprocess
from collections import Counter

%cd /content
if not os.path.exists('Calenda'):
    !git clone https://github.com/sooryong/Calenda.git
else:
    !cd Calenda && git fetch origin && git reset --hard origin/main
%cd /content/Calenda

_c = yaml.safe_load(open('configs/train_qwen3_0_6b.yaml', encoding='utf-8'))
def _stat(p):
    rows = [json.loads(l) for l in open(p, encoding='utf-8') if l.strip()]
    c = Counter(str(r['gold'].get('schedule_status')) for r in rows)
    return len(rows), c.get('yes', 0), c.get('pending', 0), c.get('no', 0)
_tt, _ty, _tp, _tn = _stat('data/processed/train.jsonl')
_gt, _gy, _gp, _gn = _stat('data/eval/golden.jsonl')
_head = subprocess.run('git log --oneline -1', shell=True, capture_output=True, text=True).stdout.strip()
print('=' * 62)
print(f"  라운드          : {_c['run_name']}  ·  epochs={_c['num_train_epochs']}")
print(f"  completion-only : {_c.get('completion_only_loss', False)}")
print(f"  커밋            : {_head[:60]}")
print(f"  학습셋          : train {_tt}   yes {_ty} / pending {_tp} / no {_tn}")
print(f"  평가셋          : golden {_gt}   yes {_gy} / pending {_gp} / no {_gn}")
print('=' * 62)

## 3. 의존성 설치

`torch`가 `+cpu`로 뜨면 런타임 유형을 GPU로 변경 후 재시작.

In [ ]:
!pip install -q -e .[train]
!pip uninstall torchao -y -q
import torch
print('torch:', torch.__version__, 'cuda:', torch.cuda.is_available(), 'ngpu:', torch.cuda.device_count())

## 4. 데이터 확인

In [ ]:
import os
for p in ['data/processed/train.jsonl', 'data/eval/golden.jsonl']:
    n = sum(1 for l in open(p, encoding='utf-8') if l.strip())
    print(f'{p}: {n} records')
_val = 'data/processed/val.jsonl'
if os.path.isfile(_val):
    print(f'{_val}: {sum(1 for l in open(_val, encoding="utf-8") if l.strip())} records')
else:
    print(f'{_val}: 없음')

## 5. 사전점검 — 학습 렌더 == 추론 프롬프트 + gold JSON

`OK` 확인 후 학습 진행.

In [ ]:
import os, yaml, json
from transformers import AutoTokenizer

CONFIG = 'configs/train_qwen3_0_6b.yaml'
_cfg  = yaml.safe_load(open(CONFIG, encoding='utf-8'))
_mcfg = yaml.safe_load(open(_cfg['model_config'], encoding='utf-8'))
print(f"라운드 {_cfg['run_name']} | base {_mcfg['hf_id']}")
print(f"output_dir {_cfg['output_dir']} | 유효배치 {_cfg['per_device_train_batch_size'] * _cfg['gradient_accumulation_steps']}")

_tok = AutoTokenizer.from_pretrained(
    _mcfg['hf_id'],
    trust_remote_code=_mcfg.get('trust_remote_code', False),
    token=False,
)
_sys  = _mcfg['system_prompt']
_user = '<채널: KakaoTalk>\n<메시지>\n내일 3시 회의\n</메시지>'
_gold = json.dumps({'schedule_status': 'no', 'events': []}, ensure_ascii=False)
_full = [{'role': 'system', 'content': _sys}, {'role': 'user', 'content': _user}, {'role': 'assistant', 'content': _gold}]
_pre  = [{'role': 'system', 'content': _sys}, {'role': 'user', 'content': _user}]
_train_str = _tok.apply_chat_template(_full, tokenize=False, add_generation_prompt=False)
_extra = {'enable_thinking': False} if 'enable_thinking' in (getattr(_tok, 'chat_template', None) or '') else {}
_infer_str = _tok.apply_chat_template(_pre, tokenize=False, add_generation_prompt=True, **_extra)
_aligned   = _train_str.startswith(_infer_str)
_json_next = (_train_str[len(_infer_str):] if _aligned else '').lstrip().startswith('{')
_rt_ok     = '<|im_start|>assistant\n' in _train_str
print(f'정렬: {_aligned} | JSON바로시작: {_json_next} | response_template: {_rt_ok}',
      '→ OK' if (_aligned and _json_next and _rt_ok) else '→ FAIL')
assert _aligned and _json_next, '학습/추론 프롬프트 정렬 깨짐'

## 6. 학습

단일 GPU · 첫 logging에서 loss 감소 확인.

In [ ]:
print(f"학습 시작 → {_cfg['run_name']} · 단일 GPU")
!python scripts/train_lora.py --config {CONFIG}

## 7. 평가 (LoRA merge → 골든셋 평가)

In [ ]:
import yaml, os
_cfg  = yaml.safe_load(open(CONFIG, encoding='utf-8'))
_mcfg = yaml.safe_load(open(_cfg['model_config'], encoding='utf-8'))

BASE       = _mcfg['hf_id']
LORA_DIR   = _cfg['output_dir']
NAME       = os.path.basename(LORA_DIR)
MERGED_DIR = f'models/merged/{NAME}'
EVAL_JSON  = f'logs/eval_{NAME}.json'
GOLDEN     = _cfg.get('eval_golden', 'data/eval/golden.jsonl')
ZIP        = f'/content/lora_{NAME}.zip'

print(f'lora: {LORA_DIR} → merged: {MERGED_DIR}')
!python scripts/merge_lora.py --base {BASE} --lora {LORA_DIR} --out {MERGED_DIR}
!python scripts/eval_model.py --model {MERGED_DIR} --eval {GOLDEN} --out {EVAL_JSON} --model_config {_cfg['model_config']}

print(f'\n--- {EVAL_JSON} ---')
print(open(EVAL_JSON, encoding='utf-8').read())

## 8. 어댑터 다운로드

세션 종료 전 즉시 다운로드. 양자화(Q8_0)는 로컬에서 `scripts/quantize.sh`로 수행.

In [ ]:
import shutil, os
from google.colab import files

shutil.make_archive(ZIP[:-4], 'zip', LORA_DIR)
print(f'zip: {ZIP} ({os.path.getsize(ZIP) // 1024 // 1024} MB)')
files.download(ZIP)
files.download(EVAL_JSON)

FAILURES_SRC = 'data/failures/round_latest.jsonl'
FAILURES_DST = f'data/failures/failures_{NAME}.jsonl'
if os.path.isfile(FAILURES_SRC):
    shutil.copy(FAILURES_SRC, FAILURES_DST)
    files.download(FAILURES_DST)
    print(f'failures: {sum(1 for l in open(FAILURES_DST) if l.strip())}건 → {FAILURES_DST}')
else:
    print('failures 없음 (전 샘플 통과)')

## 9. HF Hub 업로드 (선택 — 세션 종료 보험)

Colab Secrets에 `HF_TOKEN`이 있을 때만 실행. 세션이 끊겨도 어댑터가 HF에 보존된다.

In [ ]:
import os
from huggingface_hub import HfApi

_token = os.environ.get('HF_TOKEN', '')
if not _token:
    print('HF_TOKEN 없음 — 셀 0에서 Colab Secrets 로드 여부 확인')
else:
    HF_REPO = 'sooryong9885/Calenda-Qwen3-0.6B'   # 업로드 대상 repo
    api = HfApi(token=_token)
    api.create_repo(HF_REPO, repo_type='model', exist_ok=True)

    # LoRA 어댑터 업로드
    api.upload_folder(
        folder_path=LORA_DIR,
        repo_id=HF_REPO,
        repo_type='model',
        path_in_repo=NAME,
        commit_message=f'upload {NAME}',
    )
    print(f'업로드 완료 → https://huggingface.co/{HF_REPO}/tree/main/{NAME}')

    # eval JSON도 함께 업로드
    api.upload_file(
        path_or_fileobj=EVAL_JSON,
        path_in_repo=f'{NAME}/eval_result.json',
        repo_id=HF_REPO,
        repo_type='model',
        commit_message=f'eval result {NAME}',
    )
    print(f'eval JSON 업로드 완료')